# 10 - ViSoBERT standalone baseline

Train ViSoBERT on the ensemble split (`load_splits()`), early-stop on val. Loops pipeline seeds 42, 131, 9998 (`TM_RUN_SEEDS` override) and writes `artifacts/seeds/{seed}/metrics/visobert_standalone.json` (skip if that file exists).

Not part of the 5-model ensemble. After the loop, refreshes `artifacts/metrics/val_multiseed_summary.json` so ViSoBERT validation means appear even if notebook 07 ran first.


In [1]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy sentencepiece

In [2]:
import sys, os

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

Mounted at /content/drive
[utils_io] pulled 374 files: /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts -> /content/ensemble_tmp/artifacts
[colab_setup] is_colab=True
[colab_setup] repo_root=/content/drive/MyDrive/thesis/topicmodeling
[colab_setup] tmp_root=/content/ensemble_tmp (hf_cache/bert_work/lora_work)
[colab_setup] artifacts=/content/ensemble_tmp/artifacts
[colab_setup] persistent_artifacts=/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts
[colab_setup] HF_TOKEN: set


In [3]:
from tm_research.ensemble.utils_io import load_splits
from tm_research.ensemble.bert_oof import BertOOFConfig, run_bert_fulltrain

train_df, val_df, test_df, label_map = load_splits()
print('train', train_df.shape, 'val', val_df.shape, 'test', test_df.shape)
print('classes', label_map.class_names)

train (10549, 2) val (686, 2) test (692, 2)
classes ['Anger', 'Disgust', 'Enjoyment', 'Fear', 'Other', 'Sadness', 'Surprise']


In [4]:
from tm_research.ensemble.utils_io import (
    activate_seed_artifacts,
    reset_artifacts_root,
    push_artifacts_to_persistent,
)
from tm_research.ensemble.multiseed import (
    resolve_pipeline_seeds,
    seed_is_complete,
    write_val_multiseed_summary,
)

SKIP_FILES = ['metrics/visobert_standalone.json']
metrics = None
for seed in resolve_pipeline_seeds():
    if seed_is_complete(seed, SKIP_FILES):
        print(f'skip seed {seed}: {SKIP_FILES[0]} already present')
        continue
    activate_seed_artifacts(seed)
    try:
        print(f'Using seed: {seed}')
        cfg = BertOOFConfig(
            model_name='uitnlp/visobert',
            output_name='visobert_standalone',
            seed=seed,
            num_epochs=20,
            learning_rate=2e-5,
            max_length=256,
            train_batch_size=128,
            eval_batch_size=256,
            grad_accum_steps=2,
            warmup_ratio=0.1,
            weight_decay=0.01,
            early_stopping_patience=5,
            lr_scheduler_type='cosine',
            max_grad_norm=1.0,
            extra_tokenizer_kwargs={'use_fast': False},
            work_dir=str(paths.bert_work / f'visobert_standalone_seed_{seed}'),
        )
        metrics = run_bert_fulltrain(cfg, train_df, val_df, test_df, label_map)
    finally:
        reset_artifacts_root()

summary_path = write_val_multiseed_summary()
print('wrote', summary_path)
push_artifacts_to_persistent()


Using seed: 42
[visobert_standalone] training on full train and predicting val/test


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  471kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  390MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/692 [00:00<?, ? examples/s]

[visobert_standalone] done in 5.1 min — val=0.5889 test=0.5766
wrote /content/ensemble_tmp/artifacts/metrics/val_multiseed_summary.json
[utils_io] pushed 377 files: /content/ensemble_tmp/artifacts -> /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts


PosixPath('/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts')

In [5]:
from sklearn.metrics import classification_report

if metrics is None:
    print('No new ViSoBERT run this session (all seeds skipped). See val_multiseed_summary.json.')
else:
    print(
        f"acc={metrics['test_accuracy']:.4f}  "
        f"weighted_f1={metrics['test_f1_weighted']:.4f}  "
        f"macro_f1={metrics['test_f1']:.4f}"
    )
    print(classification_report(
        test_df['label'].map(label_map.label2id),
        metrics['test_pred'],
        target_names=label_map.class_names,
        zero_division=0,
    ))


acc=0.5766  weighted_f1=0.5789  macro_f1=0.5371
              precision    recall  f1-score   support

       Anger       0.38      0.38      0.38        40
     Disgust       0.53      0.60      0.56       131
   Enjoyment       0.74      0.60      0.66       193
        Fear       0.53      0.54      0.54        46
       Other       0.47      0.59      0.52       129
     Sadness       0.65      0.66      0.66       116
    Surprise       0.62      0.35      0.45        37

    accuracy                           0.58       692
   macro avg       0.56      0.53      0.54       692
weighted avg       0.59      0.58      0.58       692



In [6]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()

[utils_io] pushed 377 files: /content/ensemble_tmp/artifacts -> /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts


PosixPath('/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts')